<a href="https://colab.research.google.com/github/ana-sudo01/visdr/blob/vision-deteccion/entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
import torch
print("GPU disponible:", torch.cuda.is_available())

Sat Nov  7 02:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
if os.path.exists("/content/visdr"):
    %cd /content/visdr
    !git pull origin vision-deteccion
else:
    !git clone -b vision-deteccion https://github.com/ana-sudo01/visdr.git
    %cd /content/visdr
!pip install -q -r requirements.txt
!cat vision/scripts/train.py

Cloning into 'visdr'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 17 (delta 2), reused 16 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 135.81 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/visdr
import torch
from ultralytics import YOLO

dispositivo = 0 if torch.cuda.is_available() else "cpu"
print(f"entrenando en {dispositivo}")

model = YOLO("yolo11n.pt")
model.train(
    data="data/drones/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=dispositivo,
    project="runs",
    name="drones_v1",
)

In [3]:
import os
from getpass import getpass
os.environ["ROBOFLOW_API_KEY"] = getpass("Private API key de Roboflow: ").strip()
!python vision/scripts/download_dataset.py
!ls data/drones

Private API key de Roboflow: ··········
loading Roboflow workspace...
loading Roboflow project...

Extracting Dataset Version Zip to data/drones in yolov11:: 100% 19956/19956 [00:05<00:00, 3980.77it/s]
Dataset en: /content/visdr/data/drones
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [4]:
import yaml
p = "data/drones/data.yaml"
d = yaml.safe_load(open(p))
d["path"] = "/content/visdr/data/drones"
d["train"] = "train/images"
d["val"] = "valid/images"
yaml.safe_dump(d, open(p, "w"))
print(d)

{'train': 'train/images', 'val': 'valid/images', 'test': '../test/images', 'nc': 1, 'names': ['drones'], 'roboflow': {'workspace': 'drone-detection-g4d3g', 'project': 'drone-detection-a1tsf', 'version': 6, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/drone-detection-g4d3g/drone-detection-a1tsf/dataset/6'}, 'path': '/content/visdr/data/drones'}


In [5]:
!sed -i 's/epochs=30/epochs=20/' vision/scripts/train.py

In [6]:
!python vision/scripts/train.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
entrenando en 0
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/drones/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fracti

In [7]:
!find /content -name "best.pt"

/content/visdr/runs/detect/runs/drones_v1/weights/best.pt


In [8]:
from google.colab import files
files.download("runs/drones_v1/weights/best.pt")
files.download("runs/drones_v1/results.png")

FileNotFoundError: Cannot find file: runs/drones_v1/weights/best.pt